# SentimentFlow — Notebook 02: Agregaciones para Dashboard

Lee todos los datos de `reviews_sentiment` en PostgreSQL, calcula las métricas
que consume el dashboard Streamlit y las escribe en las tablas `agg_*`.

En el pipeline automático, este código se ejecuta vía Airflow (`spark/aggregations.py`)
después de cada job de análisis de sentimiento.

In [ ]:
import os

PG_HOST = os.getenv('POSTGRES_HOST', 'postgres')
PG_PORT = int(os.getenv('POSTGRES_PORT', '5432'))
PG_DB   = os.getenv('POSTGRES_DB',   'reviewsdb')
PG_USER = os.getenv('POSTGRES_USER', 'postgres')
PG_PASS = os.getenv('POSTGRES_PASSWORD', 'postgres')

print(f'PostgreSQL: {PG_HOST}:{PG_PORT}/{PG_DB}')

In [ ]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASS)
df_pg = pd.read_sql('SELECT * FROM reviews_sentiment', conn)
conn.close()

print(f'Total registros: {len(df_pg):,}')
df_pg.head(3)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName('SentimentFlow-Notebook02')
    .master('local[*]')
    .config('spark.driver.memory', '1g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

df = spark.createDataFrame(df_pg)
print(f'DataFrame Spark: {df.count()} filas')
df.printSchema()

In [ ]:
# ─── Tabla 1: métricas por producto ─────────────────────────────────────────
df_by_product = df.groupBy('product_id', 'product_name', 'category').agg(
    F.count('*').alias('total_reviews'),
    F.round(F.avg('rating'), 2).alias('avg_rating'),
    F.round(F.avg('sentiment_score'), 4).alias('avg_sentiment'),
    F.sum(F.when(F.col('sentiment_label') == 'Positivo', 1).otherwise(0)).alias('positivos'),
    F.sum(F.when(F.col('sentiment_label') == 'Neutro',   1).otherwise(0)).alias('neutros'),
    F.sum(F.when(F.col('sentiment_label') == 'Negativo', 1).otherwise(0)).alias('negativos'),
    F.max('processed_at').alias('last_updated'),
)
df_by_product.show(10)

In [ ]:
# ─── Tabla 2: evolución temporal por hora y categoría ───────────────────────
df_ts = (
    df
    .withColumn('hour_bucket', F.date_trunc('hour', F.col('event_ts')))
    .groupBy('hour_bucket', 'category')
    .agg(
        F.count('*').alias('num_reviews'),
        F.round(F.avg('sentiment_score'), 4).alias('avg_sentiment'),
        F.round(F.avg('rating'), 2).alias('avg_rating'),
    )
    .orderBy('hour_bucket')
)
df_ts.show(10)

In [ ]:
# ─── Tabla 3: sentimiento por país ──────────────────────────────────────────
df_country = (
    df.groupBy('country')
    .agg(
        F.count('*').alias('total_reviews'),
        F.round(F.avg('sentiment_score'), 4).alias('avg_sentiment'),
    )
    .orderBy('total_reviews', ascending=False)
)
df_country.show(15)

In [ ]:
import psycopg2.extras

prod_pd    = df_by_product.toPandas()
ts_pd      = df_ts.toPandas()
country_pd = df_country.toPandas()

spark.stop()

conn = psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASS)

# Upsert agg_by_product
with conn.cursor() as cur:
    for _, r in prod_pd.iterrows():
        cur.execute(
            """
            INSERT INTO agg_by_product
              (product_id, product_name, category, total_reviews, avg_rating,
               avg_sentiment, positivos, neutros, negativos, last_updated)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
            ON CONFLICT (product_id) DO UPDATE SET
              total_reviews=EXCLUDED.total_reviews, avg_rating=EXCLUDED.avg_rating,
              avg_sentiment=EXCLUDED.avg_sentiment, positivos=EXCLUDED.positivos,
              neutros=EXCLUDED.neutros, negativos=EXCLUDED.negativos,
              last_updated=EXCLUDED.last_updated
            """,
            (r.product_id, r.product_name, r.category, int(r.total_reviews),
             r.avg_rating, r.avg_sentiment, int(r.positivos), int(r.neutros),
             int(r.negativos), r.last_updated),
        )
conn.commit()
print(f'✓ agg_by_product: {len(prod_pd)} filas')

# Reemplazar agg_timeseries
with conn.cursor() as cur:
    cur.execute('TRUNCATE TABLE agg_timeseries')
    psycopg2.extras.execute_values(
        cur,
        'INSERT INTO agg_timeseries (hour_bucket, category, num_reviews, avg_sentiment, avg_rating) VALUES %s',
        [(r.hour_bucket, r.category, int(r.num_reviews), r.avg_sentiment, r.avg_rating)
         for _, r in ts_pd.iterrows()],
    )
conn.commit()
print(f'✓ agg_timeseries: {len(ts_pd)} filas')

# Upsert agg_by_country
with conn.cursor() as cur:
    for _, r in country_pd.iterrows():
        cur.execute(
            """
            INSERT INTO agg_by_country (country, total_reviews, avg_sentiment)
            VALUES (%s,%s,%s)
            ON CONFLICT (country) DO UPDATE SET
              total_reviews=EXCLUDED.total_reviews, avg_sentiment=EXCLUDED.avg_sentiment
            """,
            (r.country, int(r.total_reviews), r.avg_sentiment),
        )
conn.commit()
conn.close()
print(f'✓ agg_by_country: {len(country_pd)} filas')
print('Agregaciones completadas.')